# 15 — Build CNV consensus from canonical CopyKAT, infercnvpy, and marker annotation

This notebook is the memory-safe, repository-ready implementation of the historical final CopyKAT / infercnvpy / marker-based consensus analysis.

## Scientific semantics preserved from the historical executed notebook

The cluster-level decision rules are unchanged:

- marker-defined tumor-like clusters with CopyKAT aneuploid fraction >= 0.65 are `malignant_like_copykat_supported`;
- cycling clusters with CopyKAT aneuploid fraction >= 0.65 are `cycling_malignant_like_candidate`;
- marker-defined tumor-like clusters with CopyKAT aneuploid fraction >= 0.50 receive moderate support;
- endothelial or oligodendrocyte-like clusters with CopyKAT aneuploid fraction <= 0.25 are `non_malignant_reference_like`;
- strong marker/CopyKAT discordance is retained for review rather than forced into a malignant/non-malignant label;
- infercnvpy consensus values are appended as additional evidence and do not silently override the historical CopyKAT/marker classification rules.

## Canonical input policy

Notebook 15 consumes the standard H5AD written by Notebook 14:

`GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_microbatch.h5ad`

Notebook 14 now guarantees that its standard CopyKAT columns contain the checksum-validated archived historical CopyKAT calls used for the paper. The later CopyKAT 1.2.5 known-normal rerun is a sensitivity analysis and is not consumed here.

## Memory-safety policy

The historical notebook read the complete ~17 GB H5AD into memory. This implementation does not.

- The source H5AD is opened in backed mode.
- Only `.obs`, `X_umap`, and a few color metadata arrays are materialized.
- Summary tables and consensus labels are computed from metadata only.
- Figures are generated from a lightweight AnnData containing only `obs` and `X_umap`.
- The final H5AD is created by sequentially copying the source H5AD and replacing only `/obs`; expression matrices, layers, embeddings, and other data remain unchanged.

In [1]:
# =========================
# IMPORTS + PATHS
# =========================
from pathlib import Path
import os
import shutil
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import h5py

try:
    from anndata.io import write_elem
except Exception:
    from anndata._io.specs import write_elem


def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
            and (candidate / "metadata").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        find_repository_root(),
    )
).expanduser().resolve()

METADATA_DIR = PROJECT_DIR / "metadata"
FIG_DIR = (
    PROJECT_DIR
    / "figures"
    / "copykat_final_consensus_HVG8000_r0_6"
)
PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "integrated"
)

TAG = "HVG8000_r0_6"

COPYKAT_H5AD = (
    PROCESSED_DIR
    / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_microbatch.h5ad"
)

FALLBACK_H5AD = (
    PROCESSED_DIR
    / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status.h5ad"
)

FINAL_H5AD = (
    PROCESSED_DIR
    / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_consensus.h5ad"
)

COPYKAT_FINAL_CELL_CALLS = (
    METADATA_DIR
    / f"copykat_microbatch_final_cell_calls_{TAG}.tsv"
)

COPYKAT_CLUSTER_INTERPRETATION = (
    METADATA_DIR
    / f"copykat_microbatch_cluster_interpretation_{TAG}.tsv"
)

COPYKAT_CLUSTER_SUMMARY = (
    METADATA_DIR
    / f"copykat_microbatch_cluster_summary_{TAG}.tsv"
)

COPYKAT_OVERALL_SUMMARY = (
    METADATA_DIR
    / f"copykat_microbatch_overall_summary_{TAG}.tsv"
)

INFERCNVPY_CONSENSUS = (
    METADATA_DIR
    / f"infercnvpy_consensus_cluster_calls_{TAG}.tsv"
)

WRITE_FINAL_H5AD = True
MAKE_FIGURES = True
RANDOM_SEED = 42

CLUSTER_KEY_CANDIDATES = [
    "leiden_scvi_main",
    "leiden_scvi_HVG8000_r0_6",
    "leiden_scvi_0.6",
    "leiden_scvi_0_6",
    "leiden",
]

ANNOTATION_COL_CANDIDATES = [
    "annotation_marker_based_HVG8000_r0_6",
    "annotation_marker_based",
    "cell_type_marker_based",
    "annotation",
]

COPYKAT_PRED_CANDIDATES = [
    "copykat_microbatch_pred",
    "copykat.pred",
    "copykat_pred",
    "copykat_prediction",
]

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project:", PROJECT_DIR)
print(
    "Canonical CopyKAT H5AD exists:",
    COPYKAT_H5AD.exists(),
    COPYKAT_H5AD,
)
print(
    "Canonical CopyKAT cluster interpretation exists:",
    COPYKAT_CLUSTER_INTERPRETATION.exists(),
)
print(
    "infercnvpy consensus exists:",
    INFERCNVPY_CONSENSUS.exists(),
)


Project: G:\Repository\glioma-cnv-single-cell-atlas
Canonical CopyKAT H5AD exists: True G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_microbatch.h5ad
Canonical CopyKAT cluster interpretation exists: True
infercnvpy consensus exists: True


In [2]:
# =========================
# HELPERS
# =========================
def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    if tmp.exists():
        tmp.unlink()

    df.to_csv(
        tmp,
        sep="\t",
        index=False,
    )

    os.replace(
        tmp,
        path,
    )

    print(
        f"[SAVED] {path} shape={df.shape}"
    )

    return path


def choose_first_existing_column(
    df,
    candidates,
    required=True,
    label="column",
):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    if required:
        raise KeyError(
            f"No {label} found. Tried: {candidates}. "
            f"Available: {list(df.columns)[:80]}"
        )

    return None


def safe_numeric_cluster(value):
    if pd.isna(value):
        return np.nan

    text = str(value)

    try:
        if text.endswith(".0"):
            return int(
                float(text)
            )

        return int(text)

    except Exception:
        return text


def normalize_copykat_pred(value):
    if pd.isna(value):
        return "missing"

    text = str(
        value
    ).strip().lower()

    if text in {
        "aneuploid",
        "aneu",
        "tumor",
        "malignant",
    }:
        return "aneuploid"

    if text in {
        "diploid",
        "normal",
    }:
        return "diploid"

    if text in {
        "not.defined",
        "not_defined",
        "undefined",
        "na",
        "nan",
    }:
        return "not_defined"

    return text


def copy_file_with_progress(
    src,
    dst,
    chunk_size=64 * 1024 * 1024,
):
    src = Path(src)
    dst = Path(dst)

    total = src.stat().st_size
    copied = 0
    next_report = 5

    with src.open("rb") as fin, dst.open("wb") as fout:
        while True:
            block = fin.read(
                chunk_size
            )

            if not block:
                break

            fout.write(
                block
            )

            copied += len(block)

            pct = int(
                copied
                * 100
                / max(
                    total,
                    1,
                )
            )

            if pct >= next_report:
                print(
                    f"  {pct}% "
                    f"({copied / 1024**3:.2f} / "
                    f"{total / 1024**3:.2f} GiB)"
                )

                next_report += 5

        fout.flush()
        os.fsync(
            fout.fileno()
        )

    shutil.copystat(
        src,
        dst,
    )


def replace_h5ad_obs_group(
    h5ad_path,
    obs_df,
):
    with h5py.File(
        h5ad_path,
        "r+",
    ) as handle:

        if "obs" in handle:
            del handle["obs"]

        write_elem(
            handle,
            "obs",
            obs_df,
        )

        handle.flush()


def add_h5ad_uns_elements(
    h5ad_path,
    elements,
):
    if not elements:
        return

    with h5py.File(
        h5ad_path,
        "r+",
    ) as handle:

        if "uns" not in handle:
            handle.create_group(
                "uns"
            )

        uns_group = handle["uns"]

        for key, value in elements.items():
            if key in uns_group:
                del uns_group[key]

            write_elem(
                uns_group,
                key,
                value,
            )

        handle.flush()


def series_counts_dict(series):
    return {
        str(key): int(value)
        for key, value
        in series.value_counts(
            dropna=False
        ).items()
    }


In [3]:
# =========================
# READ CANONICAL SUMMARY TABLES + HARD INPUT CHECKPOINTS
# =========================
required_inputs = [
    COPYKAT_H5AD,
    COPYKAT_CLUSTER_INTERPRETATION,
    COPYKAT_CLUSTER_SUMMARY,
    COPYKAT_OVERALL_SUMMARY,
    INFERCNVPY_CONSENSUS,
]

missing_inputs = [
    path
    for path in required_inputs
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Notebook 15 requires the canonical outputs of Notebooks 11 and 14. "
        "Missing:\n"
        + "\n".join(
            map(
                str,
                missing_inputs,
            )
        )
    )

overall = pd.read_csv(
    COPYKAT_OVERALL_SUMMARY,
    sep="\t",
)

cluster_summary = pd.read_csv(
    COPYKAT_CLUSTER_SUMMARY,
    sep="\t",
)

cluster_interpretation = pd.read_csv(
    COPYKAT_CLUSTER_INTERPRETATION,
    sep="\t",
)

infercnvpy_consensus = pd.read_csv(
    INFERCNVPY_CONSENSUS,
    sep="\t",
)

print("CopyKAT overall:")
display(overall)

print("CopyKAT cluster interpretation:")
display(cluster_interpretation)

if cluster_interpretation.shape != (
    19,
    14,
):
    raise RuntimeError(
        "Canonical CopyKAT cluster-interpretation checkpoint failed: "
        f"{cluster_interpretation.shape} != (19, 14)"
    )

if cluster_summary.shape != (
    19,
    11,
):
    raise RuntimeError(
        "Canonical CopyKAT cluster-summary checkpoint failed: "
        f"{cluster_summary.shape} != (19, 11)"
    )

if len(overall) != 1:
    raise RuntimeError(
        "Canonical CopyKAT overall summary must contain exactly one row."
    )

expected_overall = {
    "n_prediction_files": 176,
    "n_prediction_rows_total_including_anchors": 526425,
    "n_unique_cells_in_all_prediction_rows": 351425,
    "n_target_prediction_rows": 351425,
    "n_final_cells": 351427,
    "n_final_aneuploid": 202895,
    "n_final_diploid": 111493,
    "n_final_not_defined": 37037,
    "n_final_missing": 2,
}

overall_row = overall.iloc[0]

for column, expected in expected_overall.items():
    if column not in overall.columns:
        raise RuntimeError(
            f"Canonical CopyKAT overall summary lacks required column: {column}"
        )

    observed = int(
        overall_row[column]
    )

    if observed != expected:
        raise RuntimeError(
            f"Canonical CopyKAT checkpoint failed for {column}: "
            f"{observed} != {expected}"
        )

print(
    "PASS: canonical Notebook-14 CopyKAT checkpoints match exactly."
)


CopyKAT overall:


,input_h5ad,canonical_call_archive,canonical_prediction_bundle,canonical_prediction_bundle_sha256,n_adata_cells,n_prediction_files,n_prediction_rows_total_including_anchors,n_unique_cells_in_all_prediction_rows,n_target_prediction_rows,n_final_cells,n_final_aneuploid,n_final_diploid,n_final_not_defined,n_final_missing,aneuploid_fraction_all_cells,aneuploid_fraction_defined_cells,canonical_role,rerun_policy
0,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,7b3f8c2b87847fcbcf3dff54a37dc077782e465da662cf...,351427,176,526425,351425,351425,351427,202895,111493,37037,2,0.577346,0.645365,paper-defining archived historical CopyKAT result,later CopyKAT runtimes are sensitivity analyse...


CopyKAT cluster interpretation:


,leiden_scvi_main,n_cells,n_aneuploid,n_diploid,n_not_defined,n_missing,n_defined,aneuploid_fraction_all,aneuploid_fraction_defined,not_defined_fraction,missing_fraction,annotation_marker_based_HVG8000_r0_6,copykat_cluster_call,copykat_biological_interpretation
0,4.0,32896,24754,4562,3580,0,29316,0.752493,0.844385,0.108828,0.0,putative_tumor_hypoxia_stress,copykat_aneuploid_high,marker_tumor_like_with_copykat_support
1,0.0,61646,45910,8947,6789,0,54857,0.744736,0.836903,0.110129,0.0,putative_tumor_OPC_NPC_like,copykat_aneuploid_high,marker_tumor_like_with_copykat_support
2,2.0,38634,29096,5913,3625,0,35009,0.753119,0.831101,0.093829,0.0,cycling_proliferating_cells,copykat_aneuploid_high,copykat_high_aneuploid_review_annotation
3,7.0,22524,14292,5140,3092,0,19432,0.634523,0.735488,0.137276,0.0,putative_tumor_AC_like_reactive,copykat_aneuploid_high,marker_tumor_like_with_copykat_support
4,10.0,6340,4445,1675,220,0,6120,0.701104,0.726307,0.034700,0.0,myeloid_FCN1_inflammatory_monocyte_like,copykat_aneuploid_high,copykat_high_aneuploid_review_annotation
5,11.0,6183,3646,1384,1153,0,5030,0.589681,0.724851,0.186479,0.0,putative_tumor_NPC_neuronal_like,copykat_aneuploid_high,marker_tumor_like_with_copykat_support
6,12.0,4803,2978,1446,379,0,4424,0.620029,0.673146,0.078909,0.0,MES_ECM_stromal_like_ambiguous,copykat_aneuploid_high,copykat_high_aneuploid_review_annotation
7,3.0,35354,21350,12033,1971,0,33383,0.603892,0.639547,0.055750,0.0,myeloid_inflammatory_MES_associated,copykat_aneuploid_mixed_moderate_high,review_with_marker_and_composition_context
8,1.0,59791,29402,24843,5546,0,54245,0.491746,0.542022,0.092756,0.0,myeloid_C1Q_TREM2_APOE_TAM,copykat_aneuploid_mixed_moderate_high,review_with_marker_and_composition_context
9,17.0,105,42,41,22,0,83,0.400000,0.506024,0.209524,0.0,erythroid_Hb_high_or_ambient_RBC,copykat_aneuploid_mixed_moderate_high,review_with_marker_and_composition_context


PASS: canonical Notebook-14 CopyKAT checkpoints match exactly.


In [4]:
# =========================
# READ H5AD IN BACKED MODE — OBS + UMAP ONLY
# =========================
input_h5ad = (
    COPYKAT_H5AD
    if COPYKAT_H5AD.exists()
    else FALLBACK_H5AD
)

backed = sc.read_h5ad(
    input_h5ad,
    backed="r",
)

try:
    source_shape = backed.shape

    obs = (
        backed.obs
        .copy()
    )

    obs_names = (
        backed.obs_names
        .astype(str)
        .to_numpy()
        .copy()
    )

    source_umap = (
        np.asarray(
            backed.obsm[
                "X_umap"
            ]
        ).copy()
        if "X_umap" in backed.obsm
        else None
    )

    source_color_uns = {}

    for key in [
        "annotation_marker_based_HVG8000_r0_6_colors",
        "leiden_scvi_main_colors",
    ]:
        if key in backed.uns:
            source_color_uns[key] = (
                np.asarray(
                    backed.uns[key]
                ).copy()
            )

finally:
    backed.file.close()

if source_shape != (
    351427,
    15176,
):
    raise RuntimeError(
        f"Canonical source H5AD shape mismatch: {source_shape}"
    )

obs.index = obs.index.astype(str)

if not np.array_equal(
    obs.index.to_numpy(),
    obs_names,
):
    raise RuntimeError(
        "Source H5AD obs order changed unexpectedly."
    )

cluster_col = choose_first_existing_column(
    obs,
    CLUSTER_KEY_CANDIDATES,
    label="cluster column",
)

annotation_col = choose_first_existing_column(
    obs,
    ANNOTATION_COL_CANDIDATES,
    label="annotation column",
)

copykat_col = choose_first_existing_column(
    obs,
    COPYKAT_PRED_CANDIDATES,
    required=False,
    label="CopyKAT prediction column",
)

print("Source H5AD shape:", source_shape)
print("cluster_col:", cluster_col)
print("annotation_col:", annotation_col)
print("copykat_col:", copykat_col)
print(
    "X_umap loaded:",
    None
    if source_umap is None
    else source_umap.shape,
)

if copykat_col is None:
    raise RuntimeError(
        "Canonical Notebook-14 H5AD does not contain CopyKAT calls. "
        "Do not fall back to a non-canonical rerun in Notebook 15."
    )

obs[
    "copykat_pred_clean"
] = (
    obs[
        copykat_col
    ]
    .map(
        normalize_copykat_pred
    )
    .astype(
        "category"
    )
)

obs[
    "copykat_is_defined"
] = (
    obs[
        "copykat_pred_clean"
    ]
    .isin(
        [
            "aneuploid",
            "diploid",
        ]
    )
)

obs[
    "copykat_is_aneuploid"
] = (
    obs[
        "copykat_pred_clean"
    ]
    .eq(
        "aneuploid"
    )
)

copykat_counts = series_counts_dict(
    obs[
        "copykat_pred_clean"
    ]
)

print(
    "Canonical CopyKAT cell calls:",
    copykat_counts,
)

EXPECTED_COPYKAT_CLEAN = {
    "aneuploid": 202895,
    "diploid": 111493,
    "not_defined": 37037,
    "missing": 2,
}

if copykat_counts != EXPECTED_COPYKAT_CLEAN:
    raise RuntimeError(
        "Notebook-15 source H5AD does not contain the exact canonical "
        f"historical CopyKAT calls. Observed: {copykat_counts}"
    )

print(
    "PASS: Notebook 15 is consuming the canonical historical CopyKAT H5AD."
)


Source H5AD shape: (351427, 15176)
cluster_col: leiden_scvi_main
annotation_col: annotation_marker_based_HVG8000_r0_6
copykat_col: copykat_microbatch_pred
X_umap loaded: (351427, 2)
Canonical CopyKAT cell calls: {'aneuploid': 202895, 'diploid': 111493, 'not_defined': 37037, 'missing': 2}
PASS: Notebook 15 is consuming the canonical historical CopyKAT H5AD.


In [5]:
# =========================
# BUILD HISTORICAL CLUSTER-LEVEL CONSENSUS TABLE
# =========================
ci = (
    cluster_interpretation
    .copy()
)

ci[
    "cluster"
] = (
    ci[
        "leiden_scvi_main"
    ]
    .map(
        safe_numeric_cluster
    )
    .astype(
        "object"
    )
)

ci[
    "annotation"
] = (
    ci[
        "annotation_marker_based_HVG8000_r0_6"
    ]
    .astype(
        "object"
    )
)


def classify_cluster(row):
    annotation = str(
        row.get(
            "annotation",
            "",
        )
    ).lower()

    frac_defined = row.get(
        "aneuploid_fraction_defined",
        np.nan,
    )

    n_defined = row.get(
        "n_defined",
        0,
    )

    if n_defined < 50:
        return "insufficient_cells_review"

    is_marker_tumor = annotation.startswith(
        "putative_tumor"
    )

    is_cycling = (
        "cycling"
        in annotation
    )

    is_reference_like = (
        "oligodendrocyte"
        in annotation
        or "endothelial"
        in annotation
    )

    is_clear_immune = any(
        token in annotation
        for token in [
            "myeloid",
            "t_nk",
            "b_plasma",
            "mast",
        ]
    )

    is_ambiguous = any(
        token in annotation
        for token in [
            "ambiguous",
            "erythroid",
            "hb_high",
            "mes_ecm",
            "stromal",
        ]
    )

    if (
        is_marker_tumor
        and frac_defined >= 0.65
    ):
        return "malignant_like_copykat_supported"

    if (
        is_cycling
        and frac_defined >= 0.65
    ):
        return "cycling_malignant_like_candidate"

    if (
        is_marker_tumor
        and frac_defined >= 0.50
    ):
        return "malignant_like_copykat_moderate_support"

    if (
        is_reference_like
        and frac_defined <= 0.25
    ):
        return "non_malignant_reference_like"

    if (
        is_clear_immune
        and frac_defined >= 0.60
    ):
        return "marker_copykat_discordant_high_aneuploid_review"

    if (
        is_ambiguous
        and frac_defined >= 0.60
    ):
        return "ambiguous_high_aneuploid_review"

    if frac_defined >= 0.50:
        return "mixed_moderate_aneuploid_review"

    if frac_defined < 0.35:
        return "copykat_low_or_diploid_leaning"

    return "mixed_review"


ci[
    "final_cnv_consensus_call"
] = ci.apply(
    classify_cluster,
    axis=1,
)

# Historical infercnvpy merge semantics preserved.
inf = infercnvpy_consensus.copy()

inf_cluster_col = choose_first_existing_column(
    inf,
    [
        "cluster",
        "leiden_scvi_main",
        "leiden_scvi_HVG8000_r0_6",
        "leiden_scvi_0.6",
    ],
    required=False,
    label="infercnv cluster",
)

if inf_cluster_col is not None:
    inf[
        "cluster"
    ] = (
        inf[
            inf_cluster_col
        ]
        .map(
            safe_numeric_cluster
        )
        .astype(
            "object"
        )
    )

    keep_cols = [
        "cluster"
    ] + [
        column
        for column in inf.columns
        if (
            "cnv"
            in column.lower()
            or "call"
            in column.lower()
            or "consensus"
            in column.lower()
        )
    ]

    keep_cols = list(
        dict.fromkeys(
            keep_cols
        )
    )

    ci = ci.merge(
        inf[
            keep_cols
        ].drop_duplicates(
            "cluster"
        ),
        on="cluster",
        how="left",
        suffixes=(
            "",
            "_infercnvpy",
        ),
    )

else:
    ci[
        "infercnvpy_note"
    ] = (
        "infercnvpy consensus file found but cluster column unavailable"
    )

ci[
    "cluster_str"
] = (
    ci[
        "cluster"
    ]
    .astype(str)
    .str.replace(
        r"\.0$",
        "",
        regex=True,
    )
)

FINAL_CLUSTER_CONSENSUS_TSV = (
    METADATA_DIR
    / f"final_copykat_marker_infercnvpy_cluster_consensus_{TAG}.tsv"
)

display(ci)

if ci.shape != (
    19,
    22,
):
    raise RuntimeError(
        "Historical final cluster-consensus shape checkpoint failed: "
        f"{ci.shape} != (19, 22)"
    )

EXPECTED_CLUSTER_CALLS = {
    "0": "malignant_like_copykat_supported",
    "1": "mixed_moderate_aneuploid_review",
    "2": "cycling_malignant_like_candidate",
    "3": "marker_copykat_discordant_high_aneuploid_review",
    "4": "malignant_like_copykat_supported",
    "5": "mixed_review",
    "6": "mixed_moderate_aneuploid_review",
    "7": "malignant_like_copykat_supported",
    "8": "non_malignant_reference_like",
    "9": "mixed_review",
    "10": "marker_copykat_discordant_high_aneuploid_review",
    "11": "malignant_like_copykat_supported",
    "12": "ambiguous_high_aneuploid_review",
    "13": "copykat_low_or_diploid_leaning",
    "14": "non_malignant_reference_like",
    "15": "copykat_low_or_diploid_leaning",
    "16": "copykat_low_or_diploid_leaning",
    "17": "mixed_moderate_aneuploid_review",
    "nan": "insufficient_cells_review",
}

observed_cluster_calls = dict(
    zip(
        ci[
            "cluster_str"
        ],
        ci[
            "final_cnv_consensus_call"
        ],
    )
)

if observed_cluster_calls != EXPECTED_CLUSTER_CALLS:
    audit = pd.DataFrame(
        {
            "cluster": list(
                EXPECTED_CLUSTER_CALLS.keys()
            ),
            "expected": [
                EXPECTED_CLUSTER_CALLS[key]
                for key in EXPECTED_CLUSTER_CALLS
            ],
            "observed": [
                observed_cluster_calls.get(
                    key
                )
                for key in EXPECTED_CLUSTER_CALLS
            ],
        }
    )

    audit[
        "exact_match"
    ] = (
        audit[
            "expected"
        ]
        == audit[
            "observed"
        ]
    )

    display(audit)

    raise RuntimeError(
        "Historical cluster-level consensus checkpoint failed."
    )

write_tsv_replace(
    ci,
    FINAL_CLUSTER_CONSENSUS_TSV,
)

print(
    "PASS: historical cluster-level consensus reproduced exactly."
)


,leiden_scvi_main,n_cells,n_aneuploid,n_diploid,n_not_defined,n_missing,n_defined,aneuploid_fraction_all,aneuploid_fraction_defined,not_defined_fraction,...,copykat_cluster_call,copykat_biological_interpretation,cluster,annotation,final_cnv_consensus_call,n_high_calls,n_intermediate_or_high_calls,median_of_strategy_median_cnv_abs_score,consensus_cnv_call,cluster_str
0,4.0,32896,24754,4562,3580,0,29316,0.752493,0.844385,0.108828,...,copykat_aneuploid_high,marker_tumor_like_with_copykat_support,4.0,putative_tumor_hypoxia_stress,malignant_like_copykat_supported,0.0,0.0,0.012362,CNV_low_or_reference_like,4
1,0.0,61646,45910,8947,6789,0,54857,0.744736,0.836903,0.110129,...,copykat_aneuploid_high,marker_tumor_like_with_copykat_support,0.0,putative_tumor_OPC_NPC_like,malignant_like_copykat_supported,0.0,0.0,0.016515,CNV_low_or_reference_like,0
2,2.0,38634,29096,5913,3625,0,35009,0.753119,0.831101,0.093829,...,copykat_aneuploid_high,copykat_high_aneuploid_review_annotation,2.0,cycling_proliferating_cells,cycling_malignant_like_candidate,0.0,0.0,0.016681,CNV_low_or_reference_like,2
3,7.0,22524,14292,5140,3092,0,19432,0.634523,0.735488,0.137276,...,copykat_aneuploid_high,marker_tumor_like_with_copykat_support,7.0,putative_tumor_AC_like_reactive,malignant_like_copykat_supported,0.0,0.0,0.015950,CNV_low_or_reference_like,7
4,10.0,6340,4445,1675,220,0,6120,0.701104,0.726307,0.034700,...,copykat_aneuploid_high,copykat_high_aneuploid_review_annotation,10.0,myeloid_FCN1_inflammatory_monocyte_like,marker_copykat_discordant_high_aneuploid_review,NaN,NaN,NaN,NaN,10
5,11.0,6183,3646,1384,1153,0,5030,0.589681,0.724851,0.186479,...,copykat_aneuploid_high,marker_tumor_like_with_copykat_support,11.0,putative_tumor_NPC_neuronal_like,malignant_like_copykat_supported,0.0,0.0,0.017346,CNV_low_or_reference_like,11
6,12.0,4803,2978,1446,379,0,4424,0.620029,0.673146,0.078909,...,copykat_aneuploid_high,copykat_high_aneuploid_review_annotation,12.0,MES_ECM_stromal_like_ambiguous,ambiguous_high_aneuploid_review,0.0,0.0,0.013881,CNV_low_or_reference_like,12
7,3.0,35354,21350,12033,1971,0,33383,0.603892,0.639547,0.055750,...,copykat_aneuploid_mixed_moderate_high,review_with_marker_and_composition_context,3.0,myeloid_inflammatory_MES_associated,marker_copykat_discordant_high_aneuploid_review,NaN,NaN,NaN,NaN,3
8,1.0,59791,29402,24843,5546,0,54245,0.491746,0.542022,0.092756,...,copykat_aneuploid_mixed_moderate_high,review_with_marker_and_composition_context,1.0,myeloid_C1Q_TREM2_APOE_TAM,mixed_moderate_aneuploid_review,NaN,NaN,NaN,NaN,1
9,17.0,105,42,41,22,0,83,0.400000,0.506024,0.209524,...,copykat_aneuploid_mixed_moderate_high,review_with_marker_and_composition_context,17.0,erythroid_Hb_high_or_ambient_RBC,mixed_moderate_aneuploid_review,NaN,NaN,NaN,NaN,17


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_copykat_marker_infercnvpy_cluster_consensus_HVG8000_r0_6.tsv shape=(19, 22)
PASS: historical cluster-level consensus reproduced exactly.


In [6]:
# =========================
# MAP CLUSTER CONSENSUS BACK TO CELLS
# =========================
cluster_to_call = dict(
    zip(
        ci[
            "cluster_str"
        ],
        ci[
            "final_cnv_consensus_call"
        ],
    )
)

cluster_to_frac = dict(
    zip(
        ci[
            "cluster_str"
        ],
        ci[
            "aneuploid_fraction_defined"
        ],
    )
)

cluster_to_ann = dict(
    zip(
        ci[
            "cluster_str"
        ],
        ci[
            "annotation"
        ],
    )
)

obs_cluster_str = (
    obs[
        cluster_col
    ]
    .astype(str)
    .str.replace(
        r"\.0$",
        "",
        regex=True,
    )
)

obs[
    "final_cnv_consensus_by_cluster"
] = (
    obs_cluster_str
    .map(
        cluster_to_call
    )
    .fillna(
        "unmapped_cluster"
    )
    .astype(
        "category"
    )
)

obs[
    "copykat_cluster_aneuploid_fraction_defined"
] = (
    obs_cluster_str
    .map(
        cluster_to_frac
    )
    .astype(float)
)

obs[
    "final_marker_annotation_for_cnv"
] = (
    obs_cluster_str
    .map(
        cluster_to_ann
    )
    .fillna(
        obs[
            annotation_col
        ].astype(str)
    )
    .astype(
        "category"
    )
)


def final_malignancy_from_consensus(value):
    value = str(value)

    if value in [
        "malignant_like_copykat_supported",
        "malignant_like_copykat_moderate_support",
    ]:
        return "malignant_like"

    if value == "cycling_malignant_like_candidate":
        return "cycling_malignant_like_candidate"

    if value in [
        "non_malignant_reference_like",
        "copykat_low_or_diploid_leaning",
    ]:
        return "non_malignant_or_diploid_leaning"

    if value == "unmapped_cluster":
        return "unmapped"

    return "review"


obs[
    "final_malignancy_group_copykat_marker"
] = (
    obs[
        "final_cnv_consensus_by_cluster"
    ]
    .map(
        final_malignancy_from_consensus
    )
    .astype(
        "category"
    )
)


def cell_level_group(prediction):
    prediction = normalize_copykat_pred(
        prediction
    )

    if prediction == "aneuploid":
        return "copykat_cell_aneuploid"

    if prediction == "diploid":
        return "copykat_cell_diploid"

    if prediction == "not_defined":
        return "copykat_cell_not_defined"

    return "copykat_cell_missing_or_other"


obs[
    "copykat_cell_level_group"
] = (
    obs[
        "copykat_pred_clean"
    ]
    .map(
        cell_level_group
    )
    .astype(
        "category"
    )
)

malignancy_counts = series_counts_dict(
    obs[
        "final_malignancy_group_copykat_marker"
    ]
)

consensus_counts = series_counts_dict(
    obs[
        "final_cnv_consensus_by_cluster"
    ]
)

print(
    "Final malignancy groups:",
    malignancy_counts,
)

print(
    "Final cluster-consensus cell counts:",
    consensus_counts,
)

EXPECTED_MALIGNANCY_COUNTS = {
    "review": 167178,
    "malignant_like": 123250,
    "cycling_malignant_like_candidate": 38634,
    "non_malignant_or_diploid_leaning": 22365,
}

EXPECTED_CONSENSUS_COUNTS = {
    "malignant_like_copykat_supported": 123250,
    "mixed_moderate_aneuploid_review": 82839,
    "marker_copykat_discordant_high_aneuploid_review": 41695,
    "cycling_malignant_like_candidate": 38634,
    "mixed_review": 37841,
    "non_malignant_reference_like": 17449,
    "copykat_low_or_diploid_leaning": 4916,
    "ambiguous_high_aneuploid_review": 4803,
}

if malignancy_counts != EXPECTED_MALIGNANCY_COUNTS:
    raise RuntimeError(
        "Historical final malignancy-group checkpoint failed. "
        f"Observed: {malignancy_counts}"
    )

if consensus_counts != EXPECTED_CONSENSUS_COUNTS:
    raise RuntimeError(
        "Historical final cluster-consensus cell-count checkpoint failed. "
        f"Observed: {consensus_counts}"
    )

print(
    "PASS: historical cell-level consensus counts reproduced exactly."
)


Final malignancy groups: {'review': 167178, 'malignant_like': 123250, 'cycling_malignant_like_candidate': 38634, 'non_malignant_or_diploid_leaning': 22365}
Final cluster-consensus cell counts: {'malignant_like_copykat_supported': 123250, 'mixed_moderate_aneuploid_review': 82839, 'marker_copykat_discordant_high_aneuploid_review': 41695, 'cycling_malignant_like_candidate': 38634, 'mixed_review': 37841, 'non_malignant_reference_like': 17449, 'copykat_low_or_diploid_leaning': 4916, 'ambiguous_high_aneuploid_review': 4803}
PASS: historical cell-level consensus counts reproduced exactly.


In [7]:
# =========================
# EXPORT HISTORICAL SUMMARY TABLES
# =========================
FINAL_CELL_COUNTS_TSV = (
    METADATA_DIR
    / f"final_copykat_marker_malignancy_group_counts_{TAG}.tsv"
)

FINAL_CLUSTER_COUNTS_TSV = (
    METADATA_DIR
    / f"final_copykat_marker_cluster_counts_{TAG}.tsv"
)

FINAL_ANNOTATION_COUNTS_TSV = (
    METADATA_DIR
    / f"final_copykat_marker_annotation_counts_{TAG}.tsv"
)

cell_counts = (
    obs.groupby(
        [
            "final_malignancy_group_copykat_marker",
            "copykat_pred_clean",
        ],
        observed=True,
    )
    .size()
    .reset_index(
        name="n_cells"
    )
    .sort_values(
        [
            "final_malignancy_group_copykat_marker",
            "copykat_pred_clean",
        ]
    )
)

cluster_counts = (
    obs.assign(
        cluster=obs_cluster_str
    )
    .groupby(
        [
            "cluster",
            annotation_col,
            "final_cnv_consensus_by_cluster",
            "final_malignancy_group_copykat_marker",
        ],
        observed=True,
    )
    .agg(
        n_cells=(
            "copykat_pred_clean",
            "size",
        ),
        n_copykat_aneuploid=(
            "copykat_pred_clean",
            lambda series: (
                series
                == "aneuploid"
            ).sum(),
        ),
        n_copykat_diploid=(
            "copykat_pred_clean",
            lambda series: (
                series
                == "diploid"
            ).sum(),
        ),
        n_copykat_not_defined=(
            "copykat_pred_clean",
            lambda series: (
                series
                == "not_defined"
            ).sum(),
        ),
        n_copykat_missing=(
            "copykat_pred_clean",
            lambda series: (
                series
                == "missing"
            ).sum(),
        ),
    )
    .reset_index()
)

cluster_counts[
    "n_defined"
] = (
    cluster_counts[
        "n_copykat_aneuploid"
    ]
    + cluster_counts[
        "n_copykat_diploid"
    ]
)

cluster_counts[
    "aneuploid_fraction_defined"
] = (
    cluster_counts[
        "n_copykat_aneuploid"
    ]
    / cluster_counts[
        "n_defined"
    ].replace(
        0,
        np.nan,
    )
)

cluster_counts[
    "aneuploid_fraction_all"
] = (
    cluster_counts[
        "n_copykat_aneuploid"
    ]
    / cluster_counts[
        "n_cells"
    ].replace(
        0,
        np.nan,
    )
)

cluster_counts = cluster_counts.sort_values(
    [
        "aneuploid_fraction_defined",
        "n_cells",
    ],
    ascending=[
        False,
        False,
    ],
)

annotation_counts = (
    obs.groupby(
        [
            annotation_col,
            "final_malignancy_group_copykat_marker",
        ],
        observed=True,
    )
    .size()
    .reset_index(
        name="n_cells"
    )
    .sort_values(
        [
            annotation_col,
            "n_cells",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

EXPECTED_TABLE_SHAPES = {
    "final_cluster_consensus": (19, 22),
    "final_cell_counts": (14, 3),
    "final_cluster_counts": (18, 12),
    "final_annotation_counts": (18, 3),
}

observed_table_shapes = {
    "final_cluster_consensus": ci.shape,
    "final_cell_counts": cell_counts.shape,
    "final_cluster_counts": cluster_counts.shape,
    "final_annotation_counts": annotation_counts.shape,
}

shape_audit = pd.DataFrame(
    [
        {
            "table": key,
            "expected_shape": str(
                EXPECTED_TABLE_SHAPES[
                    key
                ]
            ),
            "observed_shape": str(
                observed_table_shapes[
                    key
                ]
            ),
            "exact_match": (
                EXPECTED_TABLE_SHAPES[
                    key
                ]
                == observed_table_shapes[
                    key
                ]
            ),
        }
        for key in EXPECTED_TABLE_SHAPES
    ]
)

display(
    shape_audit
)

if not shape_audit[
    "exact_match"
].all():
    raise RuntimeError(
        "Historical Notebook-15 output-table shape checkpoint failed."
    )

write_tsv_replace(
    cell_counts,
    FINAL_CELL_COUNTS_TSV,
)

write_tsv_replace(
    cluster_counts,
    FINAL_CLUSTER_COUNTS_TSV,
)

write_tsv_replace(
    annotation_counts,
    FINAL_ANNOTATION_COUNTS_TSV,
)

display(
    cluster_counts
)

display(
    annotation_counts
)

print(
    "PASS: all historical Notebook-15 summary-table shapes match exactly."
)


,table,expected_shape,observed_shape,exact_match
0,final_cluster_consensus,"(19, 22)","(19, 22)",True
1,final_cell_counts,"(14, 3)","(14, 3)",True
2,final_cluster_counts,"(18, 12)","(18, 12)",True
3,final_annotation_counts,"(18, 3)","(18, 3)",True


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_copykat_marker_malignancy_group_counts_HVG8000_r0_6.tsv shape=(14, 3)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_copykat_marker_cluster_counts_HVG8000_r0_6.tsv shape=(18, 12)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_copykat_marker_annotation_counts_HVG8000_r0_6.tsv shape=(18, 3)


,cluster,annotation_marker_based_HVG8000_r0_6,final_cnv_consensus_by_cluster,final_malignancy_group_copykat_marker,n_cells,n_copykat_aneuploid,n_copykat_diploid,n_copykat_not_defined,n_copykat_missing,n_defined,aneuploid_fraction_defined,aneuploid_fraction_all
12,4,putative_tumor_hypoxia_stress,malignant_like_copykat_supported,malignant_like,32896,24754,4562,3580,0,29316,0.844385,0.752493
0,0,putative_tumor_OPC_NPC_like,malignant_like_copykat_supported,malignant_like,61646,45910,8947,6789,0,54857,0.836903,0.744736
10,2,cycling_proliferating_cells,cycling_malignant_like_candidate,cycling_malignant_like_candidate,38634,29096,5913,3625,0,35009,0.831101,0.753119
15,7,putative_tumor_AC_like_reactive,malignant_like_copykat_supported,malignant_like,22525,14292,5140,3092,1,19432,0.735488,0.634495
2,10,myeloid_FCN1_inflammatory_monocyte_like,marker_copykat_discordant_high_aneuploid_review,review,6340,4445,1675,220,0,6120,0.726307,0.701104
3,11,putative_tumor_NPC_neuronal_like,malignant_like_copykat_supported,malignant_like,6183,3646,1384,1153,0,5030,0.724851,0.589681
4,12,MES_ECM_stromal_like_ambiguous,ambiguous_high_aneuploid_review,review,4803,2978,1446,379,0,4424,0.673146,0.620029
11,3,myeloid_inflammatory_MES_associated,marker_copykat_discordant_high_aneuploid_review,review,35355,21350,12033,1971,1,33383,0.639547,0.603875
1,1,myeloid_C1Q_TREM2_APOE_TAM,mixed_moderate_aneuploid_review,review,59791,29402,24843,5546,0,54245,0.542022,0.491746
9,17,erythroid_Hb_high_or_ambient_RBC,mixed_moderate_aneuploid_review,review,105,42,41,22,0,83,0.506024,0.400000


,annotation_marker_based_HVG8000_r0_6,final_malignancy_group_copykat_marker,n_cells
0,B_plasma_like,non_malignant_or_diploid_leaning,1167
1,MES_ECM_stromal_like_ambiguous,review,4803
2,T_NK_cells,review,23977
3,ambiguous_mixed_Hb_T_NK_HLA,review,13864
4,cycling_proliferating_cells,cycling_malignant_like_candidate,38634
5,endothelial,non_malignant_or_diploid_leaning,2640
6,erythroid_Hb_high_or_ambient_RBC,review,105
7,mural_pericyte_fibroblast_like,non_malignant_or_diploid_leaning,3566
8,myeloid_APC_HLA_CD163_TAM,review,22943
9,myeloid_C1Q_TREM2_APOE_TAM,review,59791


PASS: all historical Notebook-15 summary-table shapes match exactly.


In [8]:
# =========================
# MAKE FIGURES FROM LIGHTWEIGHT UMAP OBJECT
# =========================
generated_color_uns = {}

if MAKE_FIGURES:
    import matplotlib.pyplot as plt

    if source_umap is None:
        print(
            "[SKIP] No X_umap in canonical source H5AD."
        )

    else:
        plot_obs = (
            obs[
                [
                    "copykat_pred_clean",
                    "final_cnv_consensus_by_cluster",
                    "final_malignancy_group_copykat_marker",
                    annotation_col,
                    cluster_col,
                ]
            ]
            .copy()
        )

        plot_adata = ad.AnnData(
            X=np.empty(
                (
                    len(plot_obs),
                    0,
                ),
                dtype=np.float32,
            ),
            obs=plot_obs,
        )

        plot_adata.obsm[
            "X_umap"
        ] = source_umap

        for key, value in source_color_uns.items():
            plot_adata.uns[
                key
            ] = value

        for color in [
            "copykat_pred_clean",
            "final_cnv_consensus_by_cluster",
            "final_malignancy_group_copykat_marker",
            annotation_col,
            cluster_col,
        ]:
            try:
                sc.pl.umap(
                    plot_adata,
                    color=color,
                    frameon=False,
                    size=2,
                    legend_loc="right margin",
                    show=False,
                )

                figure_path = (
                    FIG_DIR
                    / f"umap_{color}_{TAG}.png"
                )

                plt.savefig(
                    figure_path,
                    dpi=220,
                    bbox_inches="tight",
                )

                plt.close()

                print(
                    "[SAVED FIG]",
                    figure_path,
                )

            except Exception as error:
                print(
                    "[WARNING] could not plot",
                    color,
                    repr(error),
                )

        for key in [
            "copykat_pred_clean_colors",
            "final_cnv_consensus_by_cluster_colors",
            "final_malignancy_group_copykat_marker_colors",
        ]:
            if key in plot_adata.uns:
                generated_color_uns[
                    key
                ] = np.asarray(
                    plot_adata.uns[key]
                ).copy()

        del plot_adata

else:
    print(
        "[SKIP] MAKE_FIGURES=False"
    )


[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\copykat_final_consensus_HVG8000_r0_6\umap_copykat_pred_clean_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\copykat_final_consensus_HVG8000_r0_6\umap_final_cnv_consensus_by_cluster_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\copykat_final_consensus_HVG8000_r0_6\umap_final_malignancy_group_copykat_marker_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\copykat_final_consensus_HVG8000_r0_6\umap_annotation_marker_based_HVG8000_r0_6_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\copykat_final_consensus_HVG8000_r0_6\umap_leiden_scvi_main_HVG8000_r0_6.png


In [9]:
# =========================
# BUILD FINAL H5AD — SEQUENTIAL COPY + /obs PATCH
# =========================
if WRITE_FINAL_H5AD:
    output_obs = (
        obs
        .copy()
    )

    if not np.array_equal(
        output_obs.index.astype(str),
        obs_names,
    ):
        raise RuntimeError(
            "Final obs order does not match the canonical source H5AD."
        )

    temp_h5ad = FINAL_H5AD.with_name(
        FINAL_H5AD.stem
        + ".building.h5ad"
    )

    if temp_h5ad.exists():
        print(
            "[REMOVE STALE TEMP]",
            temp_h5ad,
        )

        temp_h5ad.unlink()

    source_size = (
        Path(
            input_h5ad
        ).stat().st_size
    )

    free_space = shutil.disk_usage(
        FINAL_H5AD.parent
    ).free

    print(
        "Source H5AD size GiB:",
        round(
            source_size
            / 1024**3,
            3,
        ),
    )

    print(
        "Free space GiB:",
        round(
            free_space
            / 1024**3,
            3,
        ),
    )

    if free_space < int(
        source_size
        * 1.05
    ):
        raise RuntimeError(
            "Insufficient free space for safe atomic final-H5AD creation."
        )

    print(
        "[COPY CANONICAL SOURCE H5AD]"
    )

    copy_file_with_progress(
        input_h5ad,
        temp_h5ad,
    )

    print(
        "[REPLACE /obs ONLY]"
    )

    replace_h5ad_obs_group(
        temp_h5ad,
        output_obs,
    )

    if generated_color_uns:
        print(
            "[ADD GENERATED COLOR METADATA]"
        )

        add_h5ad_uns_elements(
            temp_h5ad,
            generated_color_uns,
        )

    print(
        "[BACKED VALIDATION]"
    )

    check = sc.read_h5ad(
        temp_h5ad,
        backed="r",
    )

    try:
        if check.shape != (
            351427,
            15176,
        ):
            raise RuntimeError(
                f"Final H5AD shape mismatch: {check.shape}"
            )

        if not np.array_equal(
            check.obs_names.astype(str),
            output_obs.index.astype(str),
        ):
            raise RuntimeError(
                "Final H5AD obs order mismatch."
            )

        required_obs_cols = [
            "copykat_pred_clean",
            "copykat_is_defined",
            "copykat_is_aneuploid",
            "final_cnv_consensus_by_cluster",
            "copykat_cluster_aneuploid_fraction_defined",
            "final_marker_annotation_for_cnv",
            "final_malignancy_group_copykat_marker",
            "copykat_cell_level_group",
        ]

        missing_cols = [
            column
            for column in required_obs_cols
            if column not in check.obs.columns
        ]

        if missing_cols:
            raise RuntimeError(
                "Final H5AD is missing required Notebook-15 obs columns: "
                + ", ".join(
                    missing_cols
                )
            )

        final_malignancy_counts = series_counts_dict(
            check.obs[
                "final_malignancy_group_copykat_marker"
            ]
        )

        final_consensus_counts = series_counts_dict(
            check.obs[
                "final_cnv_consensus_by_cluster"
            ]
        )

        if final_malignancy_counts != EXPECTED_MALIGNANCY_COUNTS:
            raise RuntimeError(
                "Backed final-H5AD malignancy-group validation failed."
            )

        if final_consensus_counts != EXPECTED_CONSENSUS_COUNTS:
            raise RuntimeError(
                "Backed final-H5AD cluster-consensus validation failed."
            )

    finally:
        check.file.close()

    os.replace(
        temp_h5ad,
        FINAL_H5AD,
    )

    print(
        "[SAVED H5AD]",
        FINAL_H5AD,
    )

else:
    print(
        "[SKIP] WRITE_FINAL_H5AD=False"
    )


Source H5AD size GiB: 12.754
Free space GiB: 422.912
[COPY CANONICAL SOURCE H5AD]
  5% (0.69 / 12.75 GiB)
  10% (1.31 / 12.75 GiB)
  15% (1.94 / 12.75 GiB)
  20% (2.56 / 12.75 GiB)
  25% (3.25 / 12.75 GiB)
  30% (3.88 / 12.75 GiB)
  35% (4.50 / 12.75 GiB)
  40% (5.12 / 12.75 GiB)
  45% (5.75 / 12.75 GiB)
  50% (6.44 / 12.75 GiB)
  55% (7.06 / 12.75 GiB)
  60% (7.69 / 12.75 GiB)
  65% (8.31 / 12.75 GiB)
  70% (8.94 / 12.75 GiB)
  75% (9.62 / 12.75 GiB)
  80% (10.25 / 12.75 GiB)
  85% (10.88 / 12.75 GiB)
  90% (11.50 / 12.75 GiB)
  95% (12.12 / 12.75 GiB)
  100% (12.75 / 12.75 GiB)
[REPLACE /obs ONLY]
[ADD GENERATED COLOR METADATA]
[BACKED VALIDATION]
[SAVED H5AD] G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_consensus.h5ad


In [10]:
# =========================
# OUTPUT MANIFEST + FINAL HISTORICAL CHECKPOINTS
# =========================
outputs = [
    (
        "final_cluster_consensus",
        FINAL_CLUSTER_CONSENSUS_TSV,
    ),
    (
        "final_cell_counts",
        FINAL_CELL_COUNTS_TSV,
    ),
    (
        "final_cluster_counts",
        FINAL_CLUSTER_COUNTS_TSV,
    ),
    (
        "final_annotation_counts",
        FINAL_ANNOTATION_COUNTS_TSV,
    ),
]

if WRITE_FINAL_H5AD:
    outputs.append(
        (
            "final_h5ad",
            FINAL_H5AD,
        )
    )

manifest = pd.DataFrame(
    [
        {
            "output_label": label,
            "path": str(path),
            "exists": Path(
                path
            ).exists(),
            "size_mb": (
                round(
                    Path(
                        path
                    ).stat().st_size
                    / 1024**2,
                    3,
                )
                if Path(
                    path
                ).exists()
                else np.nan
            ),
        }
        for label, path in outputs
    ]
)

MANIFEST_TSV = (
    METADATA_DIR
    / f"final_copykat_marker_consensus_output_manifest_{TAG}.tsv"
)

write_tsv_replace(
    manifest,
    MANIFEST_TSV,
)

display(
    manifest
)

if WRITE_FINAL_H5AD:
    expected_manifest_shape = (
        5,
        4,
    )

else:
    expected_manifest_shape = (
        4,
        4,
    )

if manifest.shape != expected_manifest_shape:
    raise RuntimeError(
        "Notebook-15 output-manifest shape checkpoint failed: "
        f"{manifest.shape} != {expected_manifest_shape}"
    )

if not manifest[
    "exists"
].all():
    raise RuntimeError(
        "One or more Notebook-15 outputs are missing."
    )

print(
    "\nFINAL CHECKPOINTS"
)

print(
    "Canonical source H5AD:",
    input_h5ad,
)

print(
    "Source shape:",
    source_shape,
)

print(
    "CopyKAT clean counts:",
    EXPECTED_COPYKAT_CLEAN,
)

print(
    "Final cluster consensus:",
    ci.shape,
)

print(
    "Final malignancy-group counts:",
    EXPECTED_MALIGNANCY_COUNTS,
)

print(
    "Final cluster-consensus cell counts:",
    EXPECTED_CONSENSUS_COUNTS,
)

print(
    "Final cell-count table:",
    cell_counts.shape,
)

print(
    "Final cluster-count table:",
    cluster_counts.shape,
)

print(
    "Final annotation-count table:",
    annotation_counts.shape,
)

print(
    "Output manifest:",
    manifest.shape,
)

if WRITE_FINAL_H5AD:
    print(
        "Final H5AD:",
        FINAL_H5AD,
    )

print(
    "\n[DONE] Notebook 15 canonical CNV consensus completed."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_copykat_marker_consensus_output_manifest_HVG8000_r0_6.tsv shape=(5, 4)


,output_label,path,exists,size_mb
0,final_cluster_consensus,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.005
1,final_cell_counts,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
2,final_cluster_counts,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.003
3,final_annotation_counts,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
4,final_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True,13522.327



FINAL CHECKPOINTS
Canonical source H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_microbatch.h5ad
Source shape: (351427, 15176)
CopyKAT clean counts: {'aneuploid': 202895, 'diploid': 111493, 'not_defined': 37037, 'missing': 2}
Final cluster consensus: (19, 22)
Final malignancy-group counts: {'review': 167178, 'malignant_like': 123250, 'cycling_malignant_like_candidate': 38634, 'non_malignant_or_diploid_leaning': 22365}
Final cluster-consensus cell counts: {'malignant_like_copykat_supported': 123250, 'mixed_moderate_aneuploid_review': 82839, 'marker_copykat_discordant_high_aneuploid_review': 41695, 'cycling_malignant_like_candidate': 38634, 'mixed_review': 37841, 'non_malignant_reference_like': 17449, 'copykat_low_or_diploid_leaning': 4916, 'ambiguous_high_aneuploid_review': 4803}
Final cell-count table: (14, 3)
Final cluster-count table: (18, 12)
Final annotation-count table: (18, 3)
Ou